In [0]:
from pyspark.sql import Row
#this is basic code.
data = [
    Row(order_id=1, customer="Rahul", city="Mumbai",
        items=["TV", "Mobile"],
        quantities=[1,2],
        payments=[{"type":"card","amount":1500},{"type":"coupon","amount":200}]
    ),

    Row(order_id=2, customer="Anjali", city="Delhi",
        items=["Shoes","Shirt","Watch"],
        quantities=[1,1,1],
        payments=[{"type":"card","amount":800}]
    ),

    Row(order_id=3, customer="Vikram", city="Mumbai",
        items=["Laptop"],
        quantities=[1],
        payments=[{"type":"card","amount":1200},{"type":"wallet","amount":100}]
    )
]

df = spark.createDataFrame(data)

df.show(truncate=False)
df.printSchema()

In [0]:
# 1. Explode Items
# order_id | item

from pyspark.sql.functions import explode

df1 = df.select("order_id", explode("items").alias("item"))

#df1 = df1.select("order_id","item", explode("quantities").alias("quantity"))

df1.show()


In [0]:
# 2. Explode item with customer

from pyspark.sql.functions import explode

df2 = df.select("customer", explode("items").alias("items"))

df2.show()

In [0]:
# 3. Explode item with Quantities
#order_id | item | quantity

from pyspark.sql.functions import arrays_zip

df3 = df.withColumn("zipped", arrays_zip("items", "quantities")) \
       .withColumn("exploded", explode("zipped")) \
       .select(
           "order_id",
           "exploded.items",
           "exploded.quantities"
       )

df3.show()

In [0]:
# 4. Count total items per order
# order_id | total_items

from pyspark.sql.functions import size

df4 = df.select("order_id", size("items").alias("total_items"))

df4.show()


In [0]:
# 5. Orders With More Than 2 Items
# order_id | total_items

from pyspark.sql.functions import size

df5 = df.filter(size("items") > 2).select("order_id", size("items").alias("total_items"))

df5.show()


In [0]:
# 6.Extract First Item Purchased

from pyspark.sql.functions import element_at

df6 = df.select("order_id", element_at("items", 1).alias("first_item"))

df6.show()